## Structured Output — Prefill Method (claude-3-haiku)

**Technique:** Assistant prefill + stop sequences

- Pre-populate the assistant turn with ` ```json ` to force JSON output without preamble
- Use `stop_sequences=["```"]` to stop generation at the closing fence
- Result: clean JSON with no markdown or explanatory text

> **Note:** 
> - This technique requires a model that supports assistant prefill.
> - `claude-3-haiku-20240307` supports it; Claude 4.x models do not (some haiku model does).
> - All `claude-3-5-*` model variants have been deprecated.
> - Instead control the responses by system prompts

In [4]:
%pip install -q anthropic python-dotenv

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5" # Must be a 3.x model — prefill not supported on claude-sonnet-4-6

Note: you may need to restart the kernel to use updated packages.


In [5]:
messages = []

def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

system_prompt = """
    You are a helpful assistant that generates JSON responses. Always respond with only the JSON and nothing else.
    Do not include any explanations or text outside of the JSON.
    """

add_user_message(messages, "Generate a very short event bridge JSON rule for EC2 instances")

# Prefill the assistant turn — Claude will complete from here without adding preamble
add_assistant_message(messages, "```json")

# Stop generation when the closing fence is reached
result = chat(messages, system_prompt, stop_sequences=["```"])
print(result)


{
  "Name": "ec2-instance-state-change",
  "Description": "Capture EC2 instance state changes",
  "EventBusName": "default",
  "State": "ENABLED",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running", "stopped", "terminated"]
    }
  },
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:HandleEC2Event",
      "Id": "1"
    }
  ]
}



> **IMPORTANT**:
> - Normally, the message sequence will be `add_user_message` > `chat(response)` > `add_assistant_message`.
> - But when you need a structure response, it becomes `add_user_message` > `add_assistant_message` (with prefilled response) > `chat(response)` (with a stop sequence)

In [6]:
import json

# The result is already clean JSON — no fences, no text
parsed = json.loads(result)
print(json.dumps(parsed, indent=2))

{
  "Name": "ec2-instance-state-change",
  "Description": "Capture EC2 instance state changes",
  "EventBusName": "default",
  "State": "ENABLED",
  "EventPattern": {
    "source": [
      "aws.ec2"
    ],
    "detail-type": [
      "EC2 Instance State-change Notification"
    ],
    "detail": {
      "state": [
        "running",
        "stopped",
        "terminated"
      ]
    }
  },
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:HandleEC2Event",
      "Id": "1"
    }
  ]
}
